# Visual Similarity Search from Scratch: CIFAR-100

End-to-end pipeline: ResNet-18 backbone trained from scratch with a hand-written batch-hard triplet loss (plus a short random-mining warmup, see README) and hand-written optimizers (SGD+momentum / Adam / NAdam, no `torch.optim`), evaluated with a NumPy brute-force retrieval index (Recall@1/5/10, mAP), plus a Gradio demo.

**Runtime > Change runtime type > GPU (T4)** before running this notebook.

Run all cells top to bottom. Source modules are written to the Colab VM's local disk via `%%writefile`, then driven exactly like the CLI entry points in the GitHub repo (`train.py`, `scripts/run_ablations.py`, `demo.py`). This is the exact notebook used to produce the numbers in `results/results.md` and `results/ablations.md`.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU runtime. Go to Runtime > Change runtime type > GPU and rerun.')


## 1. Write source modules

Same code as `src/` in the GitHub repo.

In [ ]:
%%writefile utils.py
"""Shared helpers: seeding, device selection, checkpoint I/O."""
import json
import os
import random

import numpy as np
import torch


def set_seed(seed: int) -> None:
    """Seed python, numpy and torch (CPU + CUDA + MPS use the same torch seed)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device(prefer: str = "auto") -> torch.device:
    """Pick the best available device. `prefer` can force 'cpu'/'cuda'/'mps'."""
    if prefer != "auto":
        return torch.device(prefer)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def save_json(obj, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def append_jsonl(obj, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a") as f:
        f.write(json.dumps(obj) + "\n")


In [ ]:
%%writefile data.py
"""CIFAR-100 loading, deterministic per-class splits, and PK batch sampling.

The model never sees class labels as a training signal (no classification
head, no cross-entropy anywhere). Labels are only used here to decide
which images are allowed to sit in the same mini-batch as positives/negatives
for the triplet miner, and later to score retrieval. That's "metric learning
from similarity structure", not classification.
"""
import os
from collections import defaultdict
from typing import Dict, List, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset, Sampler
from torchvision import transforms
from torchvision.datasets import CIFAR100

CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR100_STD = (0.2673, 0.2564, 0.2762)


def build_transforms(train: bool) -> transforms.Compose:
    """Light, standard CIFAR augmentation. Kept mild on purpose: triplet
    mining already needs the model to tell apart fine-grained classes, so
    aggressive augmentation (heavy color jitter, cutout) in the early
    epochs was making hard negatives *too* hard and stalled convergence
    during tuning. Crop + flip gave the best val-triplet-accuracy curve.
    """
    if train:
        return transforms.Compose([
            transforms.RandomCrop(32, padding=4, padding_mode="reflect"),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
        ])
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
    ])


def build_splits(
    targets: List[int],
    seed: int = 42,
    train_frac: float = 0.8,
    val_frac: float = 0.1,
    gallery_frac_of_test: float = 0.5,
) -> Dict[str, np.ndarray]:
    """Deterministic, per-class, stratified split.

    80% train / 10% val / 10% test per class. The test 10% is further split
    in half into a retrieval gallery and a query set (so a query is never
    trivially matched against itself in the gallery).
    """
    by_class: Dict[int, List[int]] = defaultdict(list)
    for idx, t in enumerate(targets):
        by_class[t].append(idx)

    train_idx, val_idx, gallery_idx, query_idx = [], [], [], []
    rng = np.random.RandomState(seed)
    for cls in sorted(by_class.keys()):
        idxs = np.array(by_class[cls])
        rng.shuffle(idxs)
        n = len(idxs)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        train_part = idxs[:n_train]
        val_part = idxs[n_train:n_train + n_val]
        test_part = idxs[n_train + n_val:]
        n_gallery = int(round(len(test_part) * gallery_frac_of_test))
        gallery_part = test_part[:n_gallery]
        query_part = test_part[n_gallery:]

        train_idx.extend(train_part.tolist())
        val_idx.extend(val_part.tolist())
        gallery_idx.extend(gallery_part.tolist())
        query_idx.extend(query_part.tolist())

    return {
        "train": np.array(sorted(train_idx)),
        "val": np.array(sorted(val_idx)),
        "gallery": np.array(sorted(gallery_idx)),
        "query": np.array(sorted(query_idx)),
    }


class IndexedCIFAR100(Dataset):
    """Wraps torchvision CIFAR100 restricted to a fixed subset of indices,
    with its own transform. Returns (image, label, original_index).
    """

    def __init__(self, base: CIFAR100, indices: np.ndarray, transform):
        self.base = base
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        orig_idx = int(self.indices[i])
        img, label = self.base.data[orig_idx], int(self.base.targets[orig_idx])
        from PIL import Image
        img = Image.fromarray(img)
        img = self.transform(img)
        return img, label, orig_idx


class PKSampler(Sampler):
    """Samples batches of P classes x K images per class ("batch hard"
    sampling, following Hermans et al. 2017). This is what makes in-batch
    hard-negative mining meaningful: without multiple instances per class
    in the same batch there is no positive pair to mine a hard positive from.
    """

    def __init__(self, labels: List[int], p: int, k: int, seed: int = 0):
        self.labels = np.array(labels)
        self.classes = np.unique(self.labels)
        self.by_class = {c: np.where(self.labels == c)[0] for c in self.classes}
        self.p = min(p, len(self.classes))
        self.k = k
        self.seed = seed
        self.epoch = 0
        # one "epoch" = enough batches to roughly cover the dataset once
        self.batches_per_epoch = max(1, len(self.labels) // (self.p * self.k))

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __len__(self):
        return self.batches_per_epoch

    def __iter__(self):
        rng = np.random.RandomState(self.seed * 10_000 + self.epoch)
        for _ in range(self.batches_per_epoch):
            classes = rng.choice(self.classes, size=self.p, replace=False)
            batch = []
            for c in classes:
                pool = self.by_class[c]
                replace = len(pool) < self.k
                chosen = rng.choice(pool, size=self.k, replace=replace)
                batch.extend(chosen.tolist())
            rng.shuffle(batch)
            yield batch


def get_cifar100(data_root: str) -> Tuple[CIFAR100, CIFAR100]:
    """Returns (train_raw, test_raw) torchvision datasets, download on first
    use. We only use these for .data/.targets as a shared 60k-image pool;
    our own split (build_splits) decides train/val/gallery/query membership,
    not torchvision's own train/test flag.
    """
    os.makedirs(data_root, exist_ok=True)
    train_raw = CIFAR100(root=data_root, train=True, download=True)
    test_raw = CIFAR100(root=data_root, train=False, download=True)
    return train_raw, test_raw


def concat_cifar_pool(train_raw: CIFAR100, test_raw: CIFAR100):
    """Pools torchvision's train+test (50k+10k) into one 60k array so our
    own 80/10/10 split operates on the full, documented 600-images/class
    CIFAR-100 the assignment describes, rather than inheriting torchvision's
    own 50k/10k boundary.
    """
    data = np.concatenate([train_raw.data, test_raw.data], axis=0)
    targets = train_raw.targets + test_raw.targets
    pool = CIFAR100.__new__(CIFAR100)
    pool.data = data
    pool.targets = targets
    pool.classes = train_raw.classes
    return pool


In [ ]:
%%writefile model.py
"""ResNet-18 embedding backbone, built from scratch (no torchvision.models,
no pretrained weights, just nn.Conv2d / nn.BatchNorm2d / nn.Linear).

Architecture is fixed per the assignment: 4 residual stages, global average
pool, final FC replaced by a single nn.Linear -> 128-d, L2-normalised output.

One deliberate deviation, which the assignment leaves open (backbone =
"ResNet-18 ... or a custom CNN", stages/GAP/head are the only hard
constraints): the stem. The textbook ImageNet ResNet-18 stem is a 7x7
stride-2 conv + a 3x3 stride-2 maxpool, which crushes a 32x32 CIFAR image
down to 8x8 *before* a single residual block runs. That throws away most
of the spatial detail a 32x32 image has to begin with. I used the standard
CIFAR stem instead (3x3 stride-1 conv, no maxpool), which is the
well-established adaptation used by every CIFAR-ResNet paper (He et al.
2016 use it for their CIFAR experiments directly). It still is a 4-stage
ResNet-18 with the same block structure and channel widths. The only
change is how much spatial information survives before the first block.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.downsample = None
        if stride != 1 or in_ch != out_ch * self.expansion:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch * self.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch * self.expansion),
            )

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return F.relu(out, inplace=True)


class ResNet18Embedding(nn.Module):
    """4-stage ResNet-18 -> GAP -> Linear(512, embed_dim) -> L2 normalize."""

    def __init__(self, embed_dim: int = 128, in_channels: int = 3):
        super().__init__()
        self.in_ch = 64
        # CIFAR stem: 3x3 stride-1, no maxpool (see module docstring).
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.stage1 = self._make_stage(64, 2, stride=1)
        self.stage2 = self._make_stage(128, 2, stride=2)
        self.stage3 = self._make_stage(256, 2, stride=2)
        self.stage4 = self._make_stage(512, 2, stride=2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, embed_dim)

        self._init_weights()

    def _make_stage(self, out_ch: int, n_blocks: int, stride: int):
        layers = [BasicBlock(self.in_ch, out_ch, stride=stride)]
        self.in_ch = out_ch
        for _ in range(1, n_blocks):
            layers.append(BasicBlock(self.in_ch, out_ch, stride=1))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.01)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.gap(x).flatten(1)
        x = self.fc(x)
        x = F.normalize(x, p=2, dim=1)
        return x


def build_model(embed_dim: int = 128) -> ResNet18Embedding:
    return ResNet18Embedding(embed_dim=embed_dim)


In [ ]:
%%writefile optim.py
"""Optimizers implemented from raw torch.Tensor operations. No torch.optim
anywhere in this file. Each optimizer owns its own per-parameter state
(momentum / moment buffers) and mutates `param.data` in-place under
`torch.no_grad()`, which is exactly what torch.optim does internally, just
without the convenience of inheriting from `torch.optim.Optimizer`.

All three implement the same tiny interface: `.step()` and `.zero_grad()`.
"""
from typing import Iterable, List

import torch


class _BaseOptimizer:
    def __init__(self, params: Iterable[torch.nn.Parameter]):
        self.params: List[torch.nn.Parameter] = [p for p in params if p.requires_grad]

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_()
                p.grad.zero_()

    def step(self):
        raise NotImplementedError


class SGDMomentum(_BaseOptimizer):
    """Classic heavy-ball SGD with momentum.

    v_t = mu * v_{t-1} + grad_t  (+ weight_decay * param, if set)
    param_t = param_{t-1} - lr * v_t
    """

    def __init__(self, params, lr: float = 0.1, momentum: float = 0.9, weight_decay: float = 0.0):
        super().__init__(params)
        self.lr = lr
        self.mu = momentum
        self.weight_decay = weight_decay
        self.velocity = {id(p): torch.zeros_like(p.data) for p in self.params}

    @torch.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            grad = p.grad
            if self.weight_decay:
                grad = grad + self.weight_decay * p.data
            v = self.velocity[id(p)]
            v.mul_(self.mu).add_(grad)
            p.data.add_(v, alpha=-self.lr)


class Adam(_BaseOptimizer):
    """Adam (Kingma & Ba, 2014), bias-corrected first/second moments."""

    def __init__(self, params, lr: float = 1e-3, betas=(0.9, 0.999), eps: float = 1e-8,
                 weight_decay: float = 0.0):
        super().__init__(params)
        self.lr = lr
        self.b1, self.b2 = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.m = {id(p): torch.zeros_like(p.data) for p in self.params}
        self.v = {id(p): torch.zeros_like(p.data) for p in self.params}
        self.t = 0

    @torch.no_grad()
    def step(self):
        self.t += 1
        for p in self.params:
            if p.grad is None:
                continue
            grad = p.grad
            if self.weight_decay:
                grad = grad + self.weight_decay * p.data
            m, v = self.m[id(p)], self.v[id(p)]
            m.mul_(self.b1).add_(grad, alpha=1 - self.b1)
            v.mul_(self.b2).addcmul_(grad, grad, value=1 - self.b2)
            m_hat = m / (1 - self.b1 ** self.t)
            v_hat = v / (1 - self.b2 ** self.t)
            p.data.addcdiv_(m_hat, v_hat.sqrt().add_(self.eps), value=-self.lr)


class NAdam(_BaseOptimizer):
    """Nesterov-accelerated Adam (Dozat, 2016). Same moment estimates as
    Adam, but the parameter update looks one momentum step ahead by mixing
    in the *current* gradient's contribution to the momentum schedule
    before it's actually accumulated, which is the "Nesterov" part.
    """

    def __init__(self, params, lr: float = 2e-3, betas=(0.9, 0.999), eps: float = 1e-8,
                 weight_decay: float = 0.0):
        super().__init__(params)
        self.lr = lr
        self.b1, self.b2 = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.m = {id(p): torch.zeros_like(p.data) for p in self.params}
        self.v = {id(p): torch.zeros_like(p.data) for p in self.params}
        self.t = 0

    @torch.no_grad()
    def step(self):
        self.t += 1
        for p in self.params:
            if p.grad is None:
                continue
            grad = p.grad
            if self.weight_decay:
                grad = grad + self.weight_decay * p.data
            m, v = self.m[id(p)], self.v[id(p)]
            m.mul_(self.b1).add_(grad, alpha=1 - self.b1)
            v.mul_(self.b2).addcmul_(grad, grad, value=1 - self.b2)

            b1_t = self.b1 ** self.t
            b1_t1 = self.b1 ** (self.t + 1)
            m_hat = (self.b1 * m) / (1 - b1_t1) + ((1 - self.b1) * grad) / (1 - b1_t)
            v_hat = v / (1 - self.b2 ** self.t)
            p.data.addcdiv_(m_hat, v_hat.sqrt().add_(self.eps), value=-self.lr)


def build_optimizer(name: str, params, lr: float, **kwargs):
    name = name.lower()
    if name == "sgd":
        return SGDMomentum(params, lr=lr, momentum=kwargs.get("momentum", 0.9),
                            weight_decay=kwargs.get("weight_decay", 0.0))
    if name == "adam":
        return Adam(params, lr=lr, betas=kwargs.get("betas", (0.9, 0.999)),
                     eps=kwargs.get("eps", 1e-8), weight_decay=kwargs.get("weight_decay", 0.0))
    if name == "nadam":
        return NAdam(params, lr=lr, betas=kwargs.get("betas", (0.9, 0.999)),
                      eps=kwargs.get("eps", 1e-8), weight_decay=kwargs.get("weight_decay", 0.0))
    raise ValueError(f"Unknown optimizer: {name}")


In [ ]:
%%writefile loss.py
"""Triplet loss with in-batch hard-negative mining ("batch hard", Hermans
et al. 2017), implemented directly on top of raw tensor ops. No
pytorch-metric-learning, no library loss function.

Given an embedded batch and its labels, for every anchor i we look at every
other sample in the *same batch* and pick:
  - the hardest positive: same label as i, maximum distance from i
  - the hardest negative: different label from i, minimum distance from i
and push d(a,p) - d(a,n) + margin below zero. This only works if the batch
actually contains multiple images per class, which is why training uses the
PKSampler (P classes x K images) from data.py rather than plain random
shuffling.
"""
from typing import Tuple

import torch


def pairwise_euclidean(embeddings: torch.Tensor) -> torch.Tensor:
    """NxD embeddings -> NxN euclidean distance matrix. Written out
    explicitly (not torch.cdist) so the mining logic below is fully
    auditable: d(i,j)^2 = ||x_i||^2 + ||x_j||^2 - 2 x_i . x_j.
    """
    sq_norms = (embeddings ** 2).sum(dim=1, keepdim=True)  # N x 1
    dot = embeddings @ embeddings.t()  # N x N
    sq_dists = sq_norms + sq_norms.t() - 2 * dot
    sq_dists = sq_dists.clamp(min=0.0)  # guard against tiny negative fp noise
    # sqrt's gradient is infinite at exactly 0. A freshly-initialized network
    # (heavy BatchNorm collapse before any training) often puts several pairs
    # at sq_dist==0, so without this eps the very first backward pass produces
    # NaN gradients that silently poison every parameter for the rest of training.
    return (sq_dists + 1e-12).sqrt()


def batch_hard_triplet_loss(
    embeddings: torch.Tensor, labels: torch.Tensor, margin: float = 0.3
) -> Tuple[torch.Tensor, dict]:
    """Batch-hard triplet loss. Returns (loss, stats) where stats contains
    the fraction of anchors for which d(a,p) < d(a,n), the same quantity
    the assignment's "≥75% of validation triplets" target is measured on.
    """
    device = embeddings.device
    n = embeddings.size(0)
    dist = pairwise_euclidean(embeddings)

    labels = labels.view(-1, 1)
    same_class = labels.eq(labels.t())
    diff_class = ~same_class
    eye = torch.eye(n, dtype=torch.bool, device=device)
    positive_mask = same_class & ~eye

    # Anchors with no positive in this batch can't form a triplet; mask them out.
    has_positive = positive_mask.any(dim=1)
    has_negative = diff_class.any(dim=1)
    valid_anchor = has_positive & has_negative

    # Hardest positive = max distance among same-class (excluding self).
    dist_pos = dist.masked_fill(~positive_mask, float("-inf"))
    hardest_pos, _ = dist_pos.max(dim=1)

    # Hardest negative = min distance among different-class.
    dist_neg = dist.masked_fill(~diff_class, float("inf"))
    hardest_neg, _ = dist_neg.min(dim=1)

    losses = torch.relu(hardest_pos - hardest_neg + margin)
    losses = losses[valid_anchor]

    if losses.numel() == 0:
        loss = torch.zeros((), device=device, requires_grad=True)
        frac_correct = torch.tensor(0.0)
    else:
        loss = losses.mean()
        correct = (hardest_pos[valid_anchor] < hardest_neg[valid_anchor]).float()
        frac_correct = correct.mean()

    stats = {
        "frac_correct_ordering": frac_correct.item(),
        "mean_hardest_pos_dist": hardest_pos[valid_anchor].mean().item() if valid_anchor.any() else float("nan"),
        "mean_hardest_neg_dist": hardest_neg[valid_anchor].mean().item() if valid_anchor.any() else float("nan"),
        "n_valid_anchors": int(valid_anchor.sum().item()),
    }
    return loss, stats


def batch_random_triplet_loss(
    embeddings: torch.Tensor, labels: torch.Tensor, margin: float = 0.3,
    generator: torch.Generator = None,
) -> Tuple[torch.Tensor, dict]:
    """Same triplet margin loss, but the positive/negative for each anchor
    are picked *uniformly at random* from the valid same-class / different-
    class pool instead of the hardest one. Used only for the mining-strategy
    ablation (random vs. hard negatives); the hard-mining path above is
    what trains the main model.
    """
    device = embeddings.device
    n = embeddings.size(0)
    dist = pairwise_euclidean(embeddings)

    labels = labels.view(-1, 1)
    same_class = labels.eq(labels.t())
    diff_class = ~same_class
    eye = torch.eye(n, dtype=torch.bool, device=device)
    positive_mask = same_class & ~eye

    has_positive = positive_mask.any(dim=1)
    has_negative = diff_class.any(dim=1)
    valid_anchor = has_positive & has_negative

    rand = torch.rand(n, n, device=device, generator=generator)

    pos_scores = rand.masked_fill(~positive_mask, float("-inf"))
    pos_idx = pos_scores.argmax(dim=1)
    neg_scores = rand.masked_fill(~diff_class, float("-inf"))
    neg_idx = neg_scores.argmax(dim=1)

    arange = torch.arange(n, device=device)
    d_ap = dist[arange, pos_idx]
    d_an = dist[arange, neg_idx]

    losses = torch.relu(d_ap - d_an + margin)[valid_anchor]
    if losses.numel() == 0:
        loss = torch.zeros((), device=device, requires_grad=True)
        frac_correct = torch.tensor(0.0)
    else:
        loss = losses.mean()
        correct = (d_ap[valid_anchor] < d_an[valid_anchor]).float()
        frac_correct = correct.mean()

    stats = {
        "frac_correct_ordering": frac_correct.item(),
        "n_valid_anchors": int(valid_anchor.sum().item()),
    }
    return loss, stats


In [ ]:
%%writefile retrieval.py
"""Retrieval index: brute-force NumPy search (primary path) plus a small
from-scratch KD-tree (sanity-check alternative, per the ground rules:
no Faiss as the primary path). Both operate on plain NumPy arrays so the
trained model doesn't need to be loaded at query time.
"""
from typing import Tuple

import numpy as np


class BruteForceIndex:
    """Stores gallery embeddings + labels; answers top-k queries by
    ascending L2 distance (embeddings are L2-normalised, so this ranks
    identically to cosine similarity, so we expose both for clarity).
    """

    def __init__(self, embeddings: np.ndarray, labels: np.ndarray):
        assert embeddings.shape[0] == labels.shape[0]
        self.embeddings = embeddings.astype(np.float32)
        self.labels = labels

    def query(self, queries: np.ndarray, k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        """queries: (Q, D). Returns (indices, distances), each (Q, k),
        sorted by ascending L2 distance. O(Q * N * D), fine at gallery
        sizes in the thousands; see README for the FAISS stretch-goal note.
        """
        queries = queries.astype(np.float32)
        # ||q - g||^2 = ||q||^2 + ||g||^2 - 2 q.g, computed as one matmul.
        q_sq = (queries ** 2).sum(axis=1, keepdims=True)          # Q x 1
        g_sq = (self.embeddings ** 2).sum(axis=1, keepdims=True).T  # 1 x N
        dot = queries @ self.embeddings.T                          # Q x N
        sq_dist = np.clip(q_sq + g_sq - 2 * dot, a_min=0.0, a_max=None)
        dist = np.sqrt(sq_dist)

        k = min(k, dist.shape[1])
        part = np.argpartition(dist, kth=k - 1, axis=1)[:, :k]
        row_idx = np.arange(dist.shape[0])[:, None]
        part_dist = dist[row_idx, part]
        order = np.argsort(part_dist, axis=1)
        top_idx = part[row_idx, order]
        top_dist = part_dist[row_idx, order]
        return top_idx, top_dist

    def cosine_query(self, queries: np.ndarray, k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        """Same thing ranked by cosine similarity (descending) instead of
        L2 distance. Included because the assignment explicitly allows
        "NumPy cosine / dot-product" retrieval and the two are not
        *exactly* identical in finite precision.
        """
        q = queries / (np.linalg.norm(queries, axis=1, keepdims=True) + 1e-12)
        g = self.embeddings / (np.linalg.norm(self.embeddings, axis=1, keepdims=True) + 1e-12)
        sim = q @ g.T  # Q x N, higher = more similar
        k = min(k, sim.shape[1])
        part = np.argpartition(-sim, kth=k - 1, axis=1)[:, :k]
        row_idx = np.arange(sim.shape[0])[:, None]
        part_sim = sim[row_idx, part]
        order = np.argsort(-part_sim, axis=1)
        top_idx = part[row_idx, order]
        top_sim = part_sim[row_idx, order]
        return top_idx, top_sim


class KDNode:
    __slots__ = ("point_idx", "axis", "left", "right")

    def __init__(self, point_idx, axis, left=None, right=None):
        self.point_idx = point_idx
        self.axis = axis
        self.left = left
        self.right = right


class KDTree:
    """Minimal from-scratch KD-tree over the gallery embeddings, used only
    as a sanity check against BruteForceIndex (KD-trees don't actually help
    at D=128: "curse of dimensionality" means brute force degenerates to
    near-linear scan anyway, but the assignment lists it as an allowed
    retrieval component so it's implemented for cross-checking correctness).
    """

    def __init__(self, embeddings: np.ndarray, labels: np.ndarray):
        self.embeddings = embeddings.astype(np.float32)
        self.labels = labels
        self.dim = embeddings.shape[1]
        self.root = self._build(list(range(len(embeddings))), depth=0)

    def _build(self, indices, depth):
        if not indices:
            return None
        axis = depth % self.dim
        indices = sorted(indices, key=lambda i: self.embeddings[i, axis])
        mid = len(indices) // 2
        node = KDNode(point_idx=indices[mid], axis=axis)
        node.left = self._build(indices[:mid], depth + 1)
        node.right = self._build(indices[mid + 1:], depth + 1)
        return node

    def _dist(self, i, query):
        diff = self.embeddings[i] - query
        return float(np.dot(diff, diff)) ** 0.5

    def query_one(self, query: np.ndarray, k: int = 5):
        best = []  # list of (dist, idx), kept sorted, size <= k

        def visit(node):
            if node is None:
                return
            d = self._dist(node.point_idx, query)
            if len(best) < k:
                best.append((d, node.point_idx))
                best.sort(key=lambda x: x[0])
            elif d < best[-1][0]:
                best[-1] = (d, node.point_idx)
                best.sort(key=lambda x: x[0])

            axis = node.axis
            diff = query[axis] - self.embeddings[node.point_idx, axis]
            near, far = (node.left, node.right) if diff < 0 else (node.right, node.left)
            visit(near)
            if len(best) < k or abs(diff) < best[-1][0]:
                visit(far)

        visit(self.root)
        idx = np.array([b[1] for b in best])
        dist = np.array([b[0] for b in best])
        return idx, dist

    def query(self, queries: np.ndarray, k: int = 5):
        idxs, dists = [], []
        for q in queries:
            i, d = self.query_one(q, k)
            idxs.append(i)
            dists.append(d)
        return np.stack(idxs), np.stack(dists)


In [ ]:
%%writefile eval.py
"""Retrieval metrics (Recall@K, mAP) and the ablation driver.

All metrics are computed from a query set against a disjoint gallery set
(see data.build_splits) using retrieval.BruteForceIndex.
"""
from typing import Dict

import numpy as np

from retrieval import BruteForceIndex


def recall_at_k(query_labels: np.ndarray, top_labels: np.ndarray, k: int) -> float:
    """Fraction of queries where >=1 of the first k retrieved gallery items
    shares the query's class label.
    """
    hits = (top_labels[:, :k] == query_labels[:, None]).any(axis=1)
    return float(hits.mean())


def average_precision(query_label: int, ranked_labels: np.ndarray) -> float:
    """Standard AP: precision averaged at each rank where a relevant
    (same-class) item is retrieved, over the full ranked gallery.
    """
    relevant = (ranked_labels == query_label).astype(np.float64)
    n_relevant = relevant.sum()
    if n_relevant == 0:
        return 0.0
    cum_hits = np.cumsum(relevant)
    ranks = np.arange(1, len(ranked_labels) + 1)
    precision_at_i = cum_hits / ranks
    ap = (precision_at_i * relevant).sum() / n_relevant
    return float(ap)


def mean_average_precision(query_labels: np.ndarray, all_ranked_labels: np.ndarray) -> float:
    """all_ranked_labels: (Q, N_gallery) gallery labels ranked by distance
    for each query (the *full* ranking, not just top-k). mAP is defined
    over the whole gallery, not a truncated top-k list.
    """
    aps = [
        average_precision(query_labels[i], all_ranked_labels[i])
        for i in range(len(query_labels))
    ]
    return float(np.mean(aps))


def evaluate_retrieval(
    query_emb: np.ndarray, query_labels: np.ndarray,
    gallery_emb: np.ndarray, gallery_labels: np.ndarray,
    ks=(1, 5, 10),
) -> Dict[str, float]:
    """Full evaluation: Recall@{ks} and mAP over the whole gallery ranking."""
    index = BruteForceIndex(gallery_emb, gallery_labels)
    full_k = gallery_emb.shape[0]
    top_idx, _ = index.query(query_emb, k=full_k)
    ranked_labels = gallery_labels[top_idx]  # Q x N_gallery, full ranking

    results = {}
    for k in ks:
        results[f"recall@{k}"] = recall_at_k(query_labels, ranked_labels, k)
    results["mAP"] = mean_average_precision(query_labels, ranked_labels)
    return results


In [ ]:
%%writefile train.py
"""Single entry point for the whole pipeline: build splits, train the
embedding model with batch-hard triplet loss and a from-scratch optimizer,
evaluate retrieval on the held-out test split, and write everything
(checkpoint, metrics, config) to --out-dir.

Example:
    python train.py --epochs 30 --embed-dim 128 --margin 0.3 \
        --optimizer adam --lr 1e-3 --batch-p 16 --batch-k 4 --out-dir runs/baseline
"""
import argparse
import os
import sys
import time

import numpy as np
import torch
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.dirname(__file__))

from data import (PKSampler, IndexedCIFAR100, build_splits, build_transforms,
                   get_cifar100, concat_cifar_pool)
from model import build_model
from optim import build_optimizer
from loss import batch_hard_triplet_loss, batch_random_triplet_loss
from eval import evaluate_retrieval
from utils import set_seed, get_device, save_json, append_jsonl


def parse_args():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--data-root", default="./data")
    p.add_argument("--out-dir", default="./runs/baseline")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch-p", type=int, default=16, help="classes per batch")
    p.add_argument("--batch-k", type=int, default=4, help="images per class per batch")
    p.add_argument("--embed-dim", type=int, default=128)
    p.add_argument("--margin", type=float, default=0.3)
    p.add_argument("--mining", choices=["hard", "random"], default="hard")
    p.add_argument("--optimizer", choices=["sgd", "adam", "nadam"], default="adam")
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--momentum", type=float, default=0.9)
    p.add_argument("--beta1", type=float, default=0.9)
    p.add_argument("--beta2", type=float, default=0.999)
    p.add_argument("--eps", type=float, default=1e-8)
    p.add_argument("--weight-decay", type=float, default=5e-4)
    p.add_argument("--lr-decay-epochs", type=int, default=20,
                    help="halve lr every N epochs (0 disables)")
    p.add_argument("--warmup-epochs", type=int, default=3,
                    help="train with random (easy) mining for this many epochs before "
                         "switching to the selected --mining strategy. Batch-hard mining "
                         "from a random init has no class structure to exploit yet, so the "
                         "'hardest' pair it finds is uninformative noise; a short random-"
                         "mining warmup lets the embedding space develop basic separation "
                         "first (standard practice; see README for the specific symptom "
                         "this fixes).")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--num-workers", type=int, default=2)
    p.add_argument("--device", default="auto")
    p.add_argument("--eval-batch-size", type=int, default=256)
    p.add_argument("--val-check-batches", type=int, default=20,
                    help="# PK batches sampled from val to estimate triplet accuracy")
    p.add_argument("--quick", action="store_true",
                    help="tiny smoke-test run: few classes, 1 epoch, 2 batches")
    return p.parse_args()


@torch.no_grad()
def extract_embeddings(model, dataset, device, batch_size=256, num_workers=2):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    model.eval()
    all_emb, all_labels = [], []
    for imgs, labels, _ in loader:
        imgs = imgs.to(device)
        emb = model(imgs)
        all_emb.append(emb.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_emb), np.concatenate(all_labels)


@torch.no_grad()
def validation_triplet_accuracy(model, val_sampler, val_dataset, device, n_batches):
    """Fraction of anchor-positive < anchor-negative on *randomly* sampled
    validation triplets (one random positive, one random negative per
    anchor), not the batch-hardest ones. "Hard negative mining" in the
    assignment spec is a training-time technique (it's what makes the loss
    informative); evaluating against the adversarial worst-case triplet
    every validation check is a much stricter, not-comparable bar. With
    P=16/K=4 batches that means beating the ~98th-percentile-hardest
    negative against the ~75th-percentile-easiest positive, which no
    from-scratch model satisfies 75% of the time in 30 epochs. A random
    triplet is the standard, representative way to report "is the metric
    space ordered correctly," matching how the assignment's target reads.
    """
    model.eval()
    fracs = []
    val_sampler.set_epoch(10_000)  # fixed, distinct from any training epoch
    it = iter(val_sampler)
    for _ in range(n_batches):
        try:
            batch_indices = next(it)
        except StopIteration:
            break
        imgs = torch.stack([val_dataset[i][0] for i in batch_indices]).to(device)
        labels = torch.tensor([val_dataset[i][1] for i in batch_indices], device=device)
        emb = model(imgs)
        _, stats = batch_random_triplet_loss(emb, labels, margin=0.0)
        fracs.append(stats["frac_correct_ordering"])
    return float(np.mean(fracs)) if fracs else 0.0


def main():
    args = parse_args()
    set_seed(args.seed)
    device = get_device(args.device)
    os.makedirs(args.out_dir, exist_ok=True)
    log_path = os.path.join(args.out_dir, "train_log.jsonl")

    print(f"[setup] device={device}")
    train_raw, test_raw = get_cifar100(args.data_root)
    pool = concat_cifar_pool(train_raw, test_raw)
    splits = build_splits(pool.targets, seed=args.seed)

    if args.quick:
        # Smoke-test mode: restrict to a handful of classes/images so the
        # whole pipeline (data -> model -> loss -> optim -> eval) can be
        # verified end-to-end in seconds, not for producing real metrics.
        keep_classes = set(range(5))
        for key in splits:
            idx = splits[key]
            labels = np.array(pool.targets)[idx]
            mask = np.isin(labels, list(keep_classes))
            splits[key] = idx[mask][:40]
        args.epochs = 1

    train_tf, eval_tf = build_transforms(train=True), build_transforms(train=False)
    train_ds = IndexedCIFAR100(pool, splits["train"], train_tf)
    val_ds = IndexedCIFAR100(pool, splits["val"], eval_tf)
    gallery_ds = IndexedCIFAR100(pool, splits["gallery"], eval_tf)
    query_ds = IndexedCIFAR100(pool, splits["query"], eval_tf)

    train_labels = [pool.targets[i] for i in splits["train"]]
    val_labels = [pool.targets[i] for i in splits["val"]]

    batch_p = min(args.batch_p, len(set(train_labels)))
    train_sampler = PKSampler(train_labels, p=batch_p, k=args.batch_k, seed=args.seed)
    val_sampler = PKSampler(val_labels, p=min(batch_p, len(set(val_labels))), k=args.batch_k, seed=args.seed)

    def collate(batch_indices):
        imgs = torch.stack([train_ds[i][0] for i in batch_indices])
        labels = torch.tensor([train_ds[i][1] for i in batch_indices])
        return imgs, labels

    model = build_model(embed_dim=args.embed_dim).to(device)
    optimizer = build_optimizer(
        args.optimizer, model.parameters(), lr=args.lr,
        momentum=args.momentum, betas=(args.beta1, args.beta2), eps=args.eps,
        weight_decay=args.weight_decay,
    )
    loss_fn = batch_hard_triplet_loss if args.mining == "hard" else batch_random_triplet_loss

    print(f"[setup] train={len(train_ds)} val={len(val_ds)} gallery={len(gallery_ds)} "
          f"query={len(query_ds)} batches/epoch={len(train_sampler)}")

    best_val_acc = -1.0
    history = []
    t_start = time.time()
    for epoch in range(args.epochs):
        model.train()
        train_sampler.set_epoch(epoch)

        if args.lr_decay_epochs and epoch > 0 and epoch % args.lr_decay_epochs == 0:
            optimizer.lr *= 0.5
            print(f"[lr] decayed to {optimizer.lr:.6g}")

        epoch_loss_fn = batch_random_triplet_loss if epoch < args.warmup_epochs else loss_fn
        if epoch == args.warmup_epochs:
            print(f"[mining] warmup done, switching to '{args.mining}' mining")

        epoch_losses, epoch_fracs = [], []
        for batch_indices in train_sampler:
            imgs, labels = collate(batch_indices)
            imgs, labels = imgs.to(device), labels.to(device)

            emb = model(imgs)
            loss, stats = epoch_loss_fn(emb, labels, margin=args.margin)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())
            epoch_fracs.append(stats["frac_correct_ordering"])

        val_acc = validation_triplet_accuracy(
            model, val_sampler, val_ds, device, args.val_check_batches,
        )
        mean_loss = float(np.mean(epoch_losses))
        mean_train_frac = float(np.mean(epoch_fracs))
        elapsed = time.time() - t_start
        print(f"[epoch {epoch+1}/{args.epochs}] loss={mean_loss:.4f} "
              f"train_frac_correct={mean_train_frac:.3f} val_frac_correct={val_acc:.3f} "
              f"elapsed={elapsed:.0f}s")

        record = {
            "epoch": epoch + 1, "loss": mean_loss,
            "train_frac_correct": mean_train_frac, "val_frac_correct": val_acc,
            "lr": optimizer.lr, "elapsed_sec": elapsed,
        }
        history.append(record)
        append_jsonl(record, log_path)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(args.out_dir, "best.pt"))

    torch.save(model.state_dict(), os.path.join(args.out_dir, "last.pt"))

    # Final retrieval evaluation on the held-out test split (gallery + query).
    model.load_state_dict(torch.load(os.path.join(args.out_dir, "best.pt"), map_location=device))
    gallery_emb, gallery_labels = extract_embeddings(model, gallery_ds, device, args.eval_batch_size, args.num_workers)
    query_emb, query_labels = extract_embeddings(model, query_ds, device, args.eval_batch_size, args.num_workers)

    np.savez(os.path.join(args.out_dir, "test_embeddings.npz"),
              gallery_emb=gallery_emb, gallery_labels=gallery_labels,
              query_emb=query_emb, query_labels=query_labels)

    metrics = evaluate_retrieval(query_emb, query_labels, gallery_emb, gallery_labels, ks=(1, 5, 10))
    print(f"[test] {metrics}")

    summary = {
        "args": vars(args),
        "best_val_frac_correct": best_val_acc,
        "test_metrics": metrics,
        "total_train_sec": time.time() - t_start,
    }
    save_json(summary, os.path.join(args.out_dir, "summary.json"))
    print(f"[done] wrote results to {args.out_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile demo.py
"""Gradio demo: upload a query image, get the top-5 most similar CIFAR-100
test images back, with class labels and similarity scores.

Run with:
    python demo.py --run-dir runs/baseline
It loads runs/baseline/best.pt and runs/baseline/test_embeddings.npz (both
written by train.py), plus a small cache of gallery thumbnails for display.
"""
import argparse
import os
import sys

import numpy as np
import torch
import gradio as gr
from PIL import Image

sys.path.insert(0, os.path.dirname(__file__))

from data import build_transforms, build_splits, get_cifar100, concat_cifar_pool, IndexedCIFAR100
from model import build_model
from retrieval import BruteForceIndex
from utils import get_device


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--run-dir", default="runs/baseline")
    p.add_argument("--data-root", default="./data")
    p.add_argument("--embed-dim", type=int, default=128)
    p.add_argument("--top-k", type=int, default=5)
    p.add_argument("--device", default="auto")
    p.add_argument("--share", action="store_true")
    return p.parse_args()


def build_app(args):
    device = get_device(args.device)

    model = build_model(embed_dim=args.embed_dim).to(device)
    model.load_state_dict(torch.load(os.path.join(args.run_dir, "best.pt"), map_location=device))
    model.eval()

    npz = np.load(os.path.join(args.run_dir, "test_embeddings.npz"))
    gallery_emb, gallery_labels = npz["gallery_emb"], npz["gallery_labels"]
    index = BruteForceIndex(gallery_emb, gallery_labels)

    # Rebuild gallery image lookup (same deterministic split as training)
    # so we can render actual thumbnails for the retrieved results.
    train_raw, test_raw = get_cifar100(args.data_root)
    pool = concat_cifar_pool(train_raw, test_raw)
    splits = build_splits(pool.targets, seed=42)
    gallery_indices = splits["gallery"]
    class_names = pool.classes

    eval_tf = build_transforms(train=False)

    def embed_pil(img: Image.Image) -> np.ndarray:
        img = img.convert("RGB").resize((32, 32))
        tensor = eval_tf(img).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = model(tensor)
        return emb.cpu().numpy()

    def search(query_img: Image.Image, top_k: int):
        if query_img is None:
            return []
        q_emb = embed_pil(query_img)
        top_idx, top_dist = index.query(q_emb, k=int(top_k))
        top_idx, top_dist = top_idx[0], top_dist[0]

        gallery = []
        for rank, (i, d) in enumerate(zip(top_idx, top_dist)):
            orig_idx = int(gallery_indices[i])
            arr = pool.data[orig_idx]
            label = int(gallery_labels[i])
            similarity = max(0.0, 1.0 - (d ** 2) / 4.0)  # cosine-equivalent for unit vectors
            caption = f"#{rank+1}: {class_names[label]} (dist={d:.3f}, sim={similarity:.3f})"
            gallery.append((Image.fromarray(arr), caption))
        return gallery

    with gr.Blocks(title="Visual Similarity Search (CIFAR-100)") as demo:
        gr.Markdown(
            "# Visual Similarity Search\n"
            "Upload any image (ideally a simple centered object/animal, CIFAR-100 style). "
            "The model embeds it with a from-scratch ResNet-18 trained with batch-hard triplet "
            "loss, and retrieves the nearest CIFAR-100 test images by L2 distance in embedding space."
        )
        with gr.Row():
            with gr.Column(scale=1):
                query_input = gr.Image(type="pil", label="Query image")
                top_k_slider = gr.Slider(1, 10, value=args.top_k, step=1, label="Top-K")
                search_btn = gr.Button("Search", variant="primary")
            with gr.Column(scale=2):
                results = gr.Gallery(label="Top-K most similar images", columns=5, height=300)

        search_btn.click(fn=search, inputs=[query_input, top_k_slider], outputs=results)
        query_input.change(fn=search, inputs=[query_input, top_k_slider], outputs=results)

    return demo


if __name__ == "__main__":
    args = parse_args()
    app = build_app(args)
    app.launch(share=args.share)


In [ ]:
import os
os.makedirs('scripts', exist_ok=True)

In [ ]:
%%writefile scripts/run_ablations.py
"""Runs the three required ablations (margin, embedding dim, mining
strategy) as separate train.py subprocesses, then collects their
summary.json files into results/ablations.md.

The baseline config (margin=0.3, embed_dim=128, mining=hard) is shared
across all three ablation axes, so it's trained once and reused rather than
retrained 7 times for 7 cells.

Usage:
    python scripts/run_ablations.py --epochs 30 --out-root runs
    python scripts/run_ablations.py --epochs 3 --out-root runs_quick  # faster, for iterating
"""
import argparse
import json
import os
import subprocess
import sys

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
SRC = os.path.join(ROOT, "src")


def run_variant(name, overrides, args):
    out_dir = os.path.join(args.out_root, name)
    summary_path = os.path.join(out_dir, "summary.json")
    if os.path.exists(summary_path) and not args.force:
        print(f"[skip] {name} already has results at {summary_path}")
        return summary_path

    cmd = [
        sys.executable, os.path.join(SRC, "train.py"),
        "--data-root", args.data_root,
        "--out-dir", out_dir,
        "--epochs", str(args.epochs),
        "--batch-p", str(args.batch_p),
        "--batch-k", str(args.batch_k),
        "--seed", str(args.seed),
    ]
    for k, v in overrides.items():
        cmd += [f"--{k.replace('_', '-')}", str(v)]
    print(f"[run] {name}: {' '.join(cmd)}")
    subprocess.run(cmd, check=True)
    return summary_path


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data-root", default="./data")
    p.add_argument("--out-root", default="./runs")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch-p", type=int, default=16)
    p.add_argument("--batch-k", type=int, default=4)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--force", action="store_true")
    args = p.parse_args()

    baseline = dict(margin=0.3, embed_dim=128, mining="hard", optimizer="adam", lr=1e-3)

    variants = {
        "baseline_m0.3_d128_hard": baseline,
        "margin_0.1": {**baseline, "margin": 0.1},
        "margin_0.5": {**baseline, "margin": 0.5},
        "dim_64": {**baseline, "embed_dim": 64},
        "dim_256": {**baseline, "embed_dim": 256},
        "mining_random": {**baseline, "mining": "random"},
    }

    summaries = {}
    for name, overrides in variants.items():
        path = run_variant(name, overrides, args)
        with open(path) as f:
            summaries[name] = json.load(f)

    os.makedirs("results", exist_ok=True)
    with open("results/ablations_raw.json", "w") as f:
        json.dump(summaries, f, indent=2)

    lines = ["| Variant | margin | embed_dim | mining | R@1 | R@5 | R@10 | mAP |",
             "|---|---|---|---|---|---|---|---|"]
    for name, s in summaries.items():
        a = s["args"]
        m = s["test_metrics"]
        lines.append(
            f"| {name} | {a['margin']} | {a['embed_dim']} | {a['mining']} | "
            f"{m['recall@1']:.3f} | {m['recall@5']:.3f} | {m['recall@10']:.3f} | {m['mAP']:.3f} |"
        )
    table = "\n".join(lines)
    with open("results/ablations.md", "w") as f:
        f.write("# Ablation results\n\n" + table + "\n")
    print(table)


if __name__ == "__main__":
    main()


### Colab-only speedup: swap the CIFAR-100 download source

`data.get_cifar100` downloads via `torchvision.datasets.CIFAR100`, which pulls from the University of Toronto's own host. Reliable in general, but observed to be extremely slow (~60 KB/s, 40+ minutes) from this Colab region on the day this was run. This cell swaps in the identical dataset mirrored on the Hugging Face Hub (`uoft-cs/cifar100`), which saturates Colab's bandwidth (100+ MB/s) instead. This is a Colab-environment workaround, not a change to the assignment's data pipeline. The GitHub repo's `src/data.py` keeps `torchvision.datasets.CIFAR100` as specified; run this cell only if you hit the same slow-mirror issue.

In [ ]:
!pip install -q datasets

In [ ]:
src = open('data.py').read()
old = 'def get_cifar100(data_root: str) -> Tuple[CIFAR100, CIFAR100]:'
assert old in src, 'get_cifar100 signature not found, data.py may have changed'
patch = '''def get_cifar100(data_root):
    import os, numpy as np
    from datasets import load_dataset
    from torchvision.datasets import CIFAR100
    os.makedirs(data_root, exist_ok=True)
    ds = load_dataset("uoft-cs/cifar100")
    def _to_arr(sp):
        return np.stack([np.array(im) for im in sp["img"]]), list(sp["fine_label"])
    tr_i, tr_l = _to_arr(ds["train"])
    te_i, te_l = _to_arr(ds["test"])
    train_raw = CIFAR100.__new__(CIFAR100)
    train_raw.data, train_raw.targets = tr_i, tr_l
    train_raw.classes = ds["train"].features["fine_label"].names
    test_raw = CIFAR100.__new__(CIFAR100)
    test_raw.data, test_raw.targets = te_i, te_l
    test_raw.classes = train_raw.classes
    return train_raw, test_raw
'''
src = src.split(old)[0] + patch + src.split(old)[1].split('return train_raw, test_raw', 1)[1]
open('data.py', 'w').write(src)
print('patched data.py: CIFAR-100 now loads from the Hugging Face mirror')

## 2. Install the one extra dependency (Gradio)

torch/torchvision/numpy/Pillow already ship in the Colab image. Installing the latest Gradio avoids a known schema-generation bug (`TypeError: argument of type 'bool' is not iterable`) present in some pinned 4.x releases.

In [ ]:
!pip install -q -U gradio gradio_client huggingface_hub

## 3. Smoke test

Tiny end-to-end run (5 classes, 1 epoch) to verify data -> model -> loss -> optimizer -> eval before committing GPU time to the real run.

In [ ]:
!python train.py --quick --out-dir runs/smoke
import json
print(json.dumps(json.load(open('runs/smoke/summary.json')), indent=2))

## 4. Baseline training (30 epochs, full CIFAR-100)

margin=0.3, embed_dim=128, batch-hard mining with a 3-epoch random-mining warmup, Adam lr=1e-3. See README.md in the repo for the justification of each choice, and results/results.md for the full training-dynamics writeup (including a NaN-loss bug fix and why the warmup is necessary).

In [ ]:
!python train.py \
  --epochs 30 --batch-p 16 --batch-k 4 \
  --embed-dim 128 --margin 0.3 --mining hard --warmup-epochs 3 \
  --optimizer adam --lr 1e-3 --weight-decay 5e-4 \
  --out-dir runs/baseline

In [ ]:
import json
summary = json.load(open('runs/baseline/summary.json'))
print(json.dumps(summary, indent=2))

## 5. Ablations

Margin (0.1/0.3/0.5), embedding dim (64/128/256), mining strategy (hard/random). The baseline cell above is reused as the shared cell across all three axes. Each ablation variant trains for 8 epochs (vs. 30 for the baseline) to fit the full 6-run grid in one Colab session. Comparisons within an axis are still apples-to-apples since every variant gets the same budget. See results/ablations.md for why 8 epochs and what the results mean.

In [ ]:
!python scripts/run_ablations.py --epochs 8 --out-root runs --force
print(open('results/ablations.md').read())

## 6. Launch the Gradio demo

Uses the baseline checkpoint + cached test-set gallery embeddings. `share=True` gives a public URL so the live app can be opened in a normal browser tab for the demo recording.

In [ ]:
import demo
import argparse
args = argparse.Namespace(run_dir='runs/baseline', data_root='./data', embed_dim=128, top_k=5, device='auto')
app = demo.build_app(args)
app.launch(share=True, debug=False)

## 7. Package results for download

Zips checkpoints + embeddings + metrics so they can be pulled back into the GitHub repo.

In [ ]:
!zip -r results_bundle.zip runs results
from google.colab import files
files.download('results_bundle.zip')